# Лабораторная работа 1 — рабочий ноутбук

**ФИО: Диденко Никита Валерьевич**

**Файлы данных: S09** `clients.csv`, `transactions.csv`

Этот ноутбук — чистовой рабочий документ, а не черновик. Он должен
исполняться сверху вниз без ошибок на выданном вам **сыром** файле
и на выходе давать очищенную таблицу и `cleaning_log.csv`.

Разделы ниже обязательны, порядок менять не нужно. Внутри раздела
добавляйте столько ячеек, сколько нужно. Если какой-то раздел у вас
пустой — напишите в нём одной строкой, почему.

## 0. Окружение и настройки

In [2]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from pandas import Categorical

pd.set_option("display.width", 160)
pd.set_option("display.max_columns", 50)

RAW_CLIENTS = "clients.csv"
RAW_TXN = "transactions.csv"

## 1. Загрузка сырых данных

Файлы выгружены разными системами. Убедитесь, что они прочитались
**правильно**: кодировка, разделитель, десятичный разделитель, типы,
даты, идентификаторы.

In [8]:
df_clients_raw = pd.read_csv(RAW_CLIENTS)
print(f"clients.csv: {df_clients_raw.shape[0]} строки, {df_clients_raw.shape[1]} столбцов")

df_txn_raw = pd.read_csv(RAW_TXN, sep=";", decimal=",", encoding="cp1251")
print(f"transactions.csv: {df_txn_raw.shape[0]} строки, {df_txn_raw.shape[1]} столбцов")

display(df_clients_raw.head(3))
display(df_txn_raw.head(3))

clients.csv: 5733 строки, 16 столбцов
transactions.csv: 625604 строки, 9 столбцов


,client_id,birth_date,gender,city,segment,loyalty_tier,card_type,registration_date,card_open_date,income_source,monthly_income,has_mobile_app,app_last_login,satisfaction_score,branch_code,marketing_opt_in
0,C001526,1963-05-16,М,Санкт-Петербург,Массовый,Базовый,Кредитная,15.03.2015,06.03.2018,Зарплата в нашем банке,58900.0,Нет,NaN,3.0,335,Да
1,C002932,1955-05-17,М,Нижний Новгород,Массовый,Базовый,Дебетовая,23.05.2011,09.03.2016,Зарплата в нашем банке,53700.0,Да,2024-01,4.0,713,Да
2,C001462,1994-10-08,М,Санкт-Петербург,Зарплатный,Серебро,Дебетовая,25.04.2020,22.04.2025,Зарплата в нашем банке,54500.0,Нет,NaN,1.0,279,Нет


,txn_id,client_id,txn_datetime,amount,currency,category,mcc_code,channel,status
0,990050,C003010,2023-12-15,473.34,RUB,Связь и интернет,4814,Карта,успешно
1,990083,C003001,2023-12-17,167.44,RUB,Транспорт,4111,СБП,успешно
2,990039,C003369,2023-12-18 19:14:40,543.82,RUB,Онлайн-сервисы,5815,Карта,успешно


## 2. Аудит качества

Здесь вы ничего не исправляете — только находите и измеряете.
Для каждой метрики качества заранее определите знаменатель.

### 2.1. Объём, типы, ключи, единица наблюдения

In [21]:
print("CLIENTS")
print(df_clients_raw.info())
print("\nУникальных client_id:", df_clients_raw['client_id'].nunique(), "из", len(df_clients_raw))
print("Единица наблюдения clients.csv: клиент банка")

print("\nTRANSACTIONS")
print(df_txn_raw.info())
print("\nУникальных txn_id:", df_txn_raw['txn_id'].nunique(), "из", len(df_txn_raw))
print("Единица наблюдения transactions.csv: операция по карте")


CLIENTS
<class 'pandas.DataFrame'>
RangeIndex: 5733 entries, 0 to 5732
Data columns (total 16 columns):
 #   Column              Non-Null Count  Dtype  
---  ------              --------------  -----  
 0   client_id           5733 non-null   str    
 1   birth_date          5733 non-null   str    
 2   gender              5733 non-null   str    
 3   city                5733 non-null   str    
 4   segment             5733 non-null   str    
 5   loyalty_tier        5733 non-null   str    
 6   card_type           5733 non-null   str    
 7   registration_date   5733 non-null   str    
 8   card_open_date      5733 non-null   str    
 9   income_source       5733 non-null   str    
 10  monthly_income      5199 non-null   float64
 11  has_mobile_app      5733 non-null   str    
 12  app_last_login      3525 non-null   str    
 13  satisfaction_score  2267 non-null   float64
 14  branch_code         5733 non-null   int64  
 15  marketing_opt_in    5733 non-null   str    
dtypes: float6

### 2.2. Дубли

In [37]:
client_dups_all = df_clients_raw.duplicated(subset=['client_id'], keep=False)
n_client_dups = int(df_clients_raw['client_id'].duplicated().sum())

print(f"Для clients.csv: строк с повторяющимся client_id: {client_dups_all.sum()} ({n_client_dups} дублирующих записей),")
print(f"полных строк-дубликатов: {df_clients_raw.duplicated().sum()}.")

txn_dups_all = df_txn_raw.duplicated(subset=['txn_id'], keep=False)
n_txn_dups = int(df_txn_raw['txn_id'].duplicated().sum())

print(f"\nДля transactions.csv: строк с повторяющимся txn_id: {txn_dups_all.sum()} ({n_txn_dups} дублирующих записей),")
print(f"полных строк-дубликатов: {df_txn_raw.duplicated().sum()}.")

Для clients.csv: строк с повторяющимся client_id: 109 (55 дублирующих записей),
полных строк-дубликатов: 40.

Для transactions.csv: строк с повторяющимся txn_id: 3000 (1500 дублирующих записей),
полных строк-дубликатов: 1281.


In [36]:
clients_no_full = df_clients_raw.drop_duplicates()
partial_client_dups = clients_no_full[
    clients_no_full.duplicated(subset=['client_id'], keep=False)
].sort_values('client_id')

print(
    f'Неполных дубликатов в clients: {len(partial_client_dups)} строк ({partial_client_dups["client_id"].nunique()} уникальных клиентов)'
)
display(
    partial_client_dups[
        ['client_id', 'city', 'segment', 'branch_code', 'monthly_income']
    ].head(10)
)

txn_no_full = df_txn_raw.drop_duplicates()
partial_txn_dups = txn_no_full[
    txn_no_full.duplicated(subset=['txn_id'], keep=False)
].sort_values('txn_id')

print(
    f'Неполных дубликатов в transactions: {len(partial_txn_dups)} строк ({partial_txn_dups["txn_id"].nunique()} уникальных транзакций)'
)
display(
    partial_txn_dups[
        ['txn_id', 'client_id', 'txn_datetime', 'amount', 'category', 'status']
    ].head(10)
)

Неполных дубликатов в clients: 30 строк (15 уникальных клиентов)


,client_id,city,segment,branch_code,monthly_income
4097,C000165,Москва,Массовый,360,35700.0
2170,C000165,Казань,Массовый,360,35700.0
5574,C000389,Москва,Массовый,808,50300.0
567,C000389,Москва,Зарплатный,808,50300.0
1469,C000652,Москва,Массовый,348,NaN
1642,C000652,Н. Новгород,Премиальный,348,NaN
173,C001381,г. Санкт-Петербург,Массовый,110,45700.0
411,C001381,Санкт-Петербург,Private,110,45700.0
1655,C001464,Новосибирск,Зарплатный,837,64100.0
3462,C001464,москва,Массовый,837,64100.0


Неполных дубликатов в transactions: 438 строк (219 уникальных транзакций)


,txn_id,client_id,txn_datetime,amount,category,status
5101,105087,C002333,2024-01-06,1058.46,Развлечения и досуг,успешно
5102,105087,C002333,2024-01-06 14:02:11,1058.46,Развлечения и досуг,успешно
5370,105355,C001585,2024-01-06 15:36:51,1797.00,Развлечения и досуг,успешно
5371,105355,C001585,2024-01-06,1797.00,Развлечения и досуг,успешно
6152,106133,C001910,2024-01-07 10:22:46,1160.68,Связь и Интернет,успешно
6153,106133,C001910,2024-01-07,1160.68,Связь и Интернет,успешно
7356,107334,C000239,2024-01-08 16:51:52,1431.49,Развлечения и досуг,успешно
7357,107334,C000239,2024-01-08,1431.49,Развлечения и досуг,успешно
8632,108607,C005274,2024-01-10,3068.23,Продукты,успешно
8633,108607,C005274,2024-01-10 14:07:54,3068.23,Продукты,успешно


### 2.3. Соответствие данных описанию полей из задания

In [47]:
CLIENTS_CATEGORICAL_COLS = ['gender', 'city', 'segment', 'loyalty_tier', 'card_type', 'has_mobile_app', 'marketing_opt_in']

for col in CLIENTS_CATEGORICAL_COLS:
    display(df_clients_raw[col].value_counts(dropna=False).to_frame('count'))

invalid_income = df_clients_raw[df_clients_raw['monthly_income'] < 0]

print(f'Найдено записей с monthly_income <= 0: {len(invalid_income)}')
display(
    invalid_income[[
        'client_id',
        'monthly_income',
        'segment',
        'city',
        'income_source',
        'branch_code',
    ]].head(10)
)

,count
gender,
М,2908
Ж,2825


,count
city,
Москва,1092
Нижний Новгород,790
Санкт-Петербург,672
Казань,552
Новосибирск,550
Екатеринбург,480
Н. Новгород,203
г. Новосибирск,194
г. Москва,185


,count
segment,
Массовый,2935
Зарплатный,1656
Премиальный,895
Private,247


,count
loyalty_tier,
Базовый,2908
Серебро,1543
Золото,941
Платина,341


,count
card_type,
Дебетовая,4053
Кредитная,1176
Премиальная,504


,count
has_mobile_app,
Да,3525
Нет,2208


,count
marketing_opt_in,
Да,5045
Нет,688


Найдено записей с monthly_income <= 0: 42


,client_id,monthly_income,segment,city,income_source,branch_code
231,C005358,-999.0,Зарплатный,г. Москва,Зарплата в нашем банке,207
253,C004293,-999.0,Массовый,г. Новосибирск,Зарплата в другом банке,318
406,C002269,-999.0,Массовый,Екатеринбург,Не указан,348
437,C000873,-999.0,Зарплатный,Казань,Зарплата в нашем банке,863
448,C003398,-999.0,Массовый,г. Новосибирск,Зарплата в другом банке,441
644,C004915,-999.0,Зарплатный,Новосибирск,Зарплата в нашем банке,265
651,C001841,-999.0,Массовый,Нижний Новгород,Зарплата в другом банке,276
678,C003082,-999.0,Массовый,Казань,Не указан,668
695,C005339,-999.0,Массовый,Москва,Зарплата в другом банке,477
1224,C002463,-999.0,Массовый,Новосибирск,Зарплата в другом банке,866


In [43]:
TXN_CATEGORICAL_COLS = ['currency', 'category', 'channel', 'status']

for col in TXN_CATEGORICAL_COLS:
    display(df_txn_raw[col].value_counts(dropna=False).to_frame('count'))

txn_dt_temp = pd.to_datetime(df_txn_raw['txn_datetime'], format='mixed')
print(f"\nДиапазон дат: с {txn_dt_temp.min()} по {txn_dt_temp.max()}")
print(f"Операций до 2024-01-01: {(txn_dt_temp < '2024-01-01').sum()}")
print(f"Операций после 2025-06-30: {(txn_dt_temp > '2025-06-30 23:59:59').sum()}")

,count
currency,
RUB,623129
USD,1251
EUR,1224


,count
category,
Продукты,165139
Транспорт,65761
Одежда и обувь,64493
АЗС,58127
Развлечения и досуг,53697
Кафе и рестораны,48234
Аптеки,45480
Онлайн-сервисы,43855
Связь и интернет,29465


,count
channel,
Карта,319523
СБП,108166
Токен,105570
Онлайн-банк,92345


,count
status,
успешно,605722
отклонено,12240
возврат,7642



Диапазон дат: с 2023-12-15 00:00:00 по 2026-08-04 13:29:31
Операций до 2024-01-01: 9
Операций после 2025-06-30: 88


### 2.4. Связь между таблицами

In [33]:
cids_clients = set(df_clients_raw['client_id'])
cids_txns = set(df_txn_raw['client_id'])

diff_tx_only = cids_txns - cids_clients
diff_cl_only = cids_clients - cids_txns

print(f"Клиентов в clients.csv: {len(cids_clients)}")
print(f"Клиентов в transactions.csv: {len(cids_txns)}")
print(f"Клиентов есть в transactions.csv, но нет в clients.csv: {len(diff_tx_only)} (совершили {df_txn_raw['client_id'].isin(diff_tx_only).sum()} транзакций)")
print(f"Клиентов есть в clients.csv, но нет в transactions.csv: {len(diff_cl_only)}")

Клиентов в clients.csv: 5678
Клиентов в transactions.csv: 5990
Клиентов есть в transactions.csv, но нет в clients.csv: 322 (совершили 26941 транзакций)
Клиентов есть в clients.csv, но нет в transactions.csv: 10


**Выводы по аудиту.**

- в обеих таблицах присутствуют дубликаты по id клиента (55 дублирующих записей, из которых 40 полных дубликатов) и по id транзакции (1500 дублирующих записей, из которых 1281 полный дубликат),
- в `transactions.csv` есть записи вне исследуемого периода,
- в `monthly_income` существуют записи с отрицательным доходом (равным -999),
- в `transactions.csv` есть клиенты, которых нет в `clients.csv` (необходимо удалить),
- в `city` существуют разные варианты записи городов (17 различных записей для 6 городов),
- в `category` существуют разные варианты записи категорий трат.

## 3. Препроцессинг

На каждое изменение — строка в журнале (раздел 4). Решение «оставить
как есть» тоже является решением и тоже попадает в журнал.

### 3.1. Типы, даты, числа

In [38]:
clients_clean = df_clients_raw.copy()
txn_clean = df_txn_raw.copy()

# Приведение дат
clients_clean['birth_date_dt'] = pd.to_datetime(clients_clean['birth_date'], format='%Y-%m-%d', errors='coerce')
clients_clean['registration_date_dt'] = pd.to_datetime(clients_clean['registration_date'], format='%d.%m.%Y', errors='coerce')
clients_clean['card_open_date_dt'] = pd.to_datetime(clients_clean['card_open_date'], format='%d.%m.%Y', errors='coerce')
clients_clean['age'] = (pd.to_datetime('2025-06-30') - clients_clean['birth_date_dt']).dt.days // 365.25


# Приведение дат в transactions
txn_clean['txn_dt'] = pd.to_datetime(txn_clean['txn_datetime'], format='mixed')
print("Приведение типов и дат завершено.")

Приведение типов и дат завершено.


### 3.2. Категориальные признаки

In [42]:
# Нормализация городов
city_mapping = {
    'Москва': 'Москва', 'г. Москва': 'Москва', ' МОСКВА ': 'Москва', ' москва': 'Москва', 'МОСКВА': 'Москва', 'москва': 'Москва',
    'Санкт-Петербург': 'Санкт-Петербург', 'СПб': 'Санкт-Петербург', 'г. Санкт-Петербург': 'Санкт-Петербург', 'Санкт Петербург': 'Санкт-Петербург',
    'Нижний Новгород': 'Нижний Новгород', 'Н. Новгород': 'Нижний Новгород',
    'Казань': 'Казань', 'казань': 'Казань',
    'Новосибирск': 'Новосибирск', 'г. Новосибирск': 'Новосибирск',
    'Екатеринбург': 'Екатеринбург', 'ЕКАТЕРИНБУРГ': 'Екатеринбург', 'г.Екатеринбург': 'Екатеринбург'
}
n_city_variants = int((clients_clean['city'] != clients_clean['city'].str.strip().map(city_mapping)).sum())
clients_clean['city_norm'] = clients_clean['city'].str.strip().map(lambda x: city_mapping.get(x, x))
print(f"Нормализовано строк городов: {n_city_variants}. Распределение городов:\n", clients_clean['city_norm'].value_counts())

# Нормализация категорий трат
cat_mapping = {
    'Кафе/рестораны': 'Кафе и рестораны',
    'кафе и рестораны': 'Кафе и рестораны',
    'Связь и Интернет': 'Связь и интернет'
}
n_cat_changed = int(txn_clean['category'].isin(cat_mapping.keys()).sum())
txn_clean['category_norm'] = txn_clean['category'].replace(cat_mapping)
print(f"\nНормализовано категорий транзакций: {n_cat_changed}. Распределение категорий:\n", txn_clean['category_norm'].value_counts())

Нормализовано строк городов: 1597. Распределение городов:
 city_norm
Москва             1629
Санкт-Петербург    1000
Нижний Новгород     993
Новосибирск         744
Казань              688
Екатеринбург        679
Name: count, dtype: int64

Нормализовано категорий транзакций: 51353. Распределение категорий:
 category_norm
Продукты               165139
Кафе и рестораны        80236
Транспорт               65761
Одежда и обувь          64493
АЗС                     58127
Развлечения и досуг     53697
Связь и интернет        48816
Аптеки                  45480
Онлайн-сервисы          43855
Name: count, dtype: int64


### 3.3. Пропуски

Для каждого поля с пропусками: доля, предположение о механизме, решение.

In [ ]:


# Замена -999 на NaN в monthly_income
clients_clean['monthly_income_clean'] = clients_clean['monthly_income'].replace(-999.0, np.nan)

### 3.4. Выбросы

Обнаружение и решение — это два разных шага. Сначала первый, потом второй.

### 3.5. Фильтры, применяемые перед расчётом активности

Покажите воронку: сколько строк на входе и сколько осталось после каждого фильтра.

## 4. Журнал изменений

Одна строка — одно решение, а не одна запись. Колонки: `field`, `issue`,
`scope`, `n_rows`, `value_before`, `action`, `value_after`, `rationale`.

In [16]:
cleaning_log = pd.DataFrame(columns=[
    "field", "issue", "scope", "n_rows",
    "value_before", "action", "value_after", "rationale",
])

## 5. Разведочный анализ

Графиков столько, сколько нужно. В отчёт пойдут не все.

### 5.1. Распределения количественных признаков

### 5.2. Частотные характеристики категориальных признаков

### 5.3. Динамика по месяцам

### 5.4. Разрезы и связи между признаками

## 6. Метрика активности

Что именно вы считаете активностью, как она вычисляется, на каком
множестве клиентов, за какие окна. Отдельно: какое снижение эта метрика
**не увидит**.

## 7. Поиск и описание групп

Правило отбора для каждой группы, её размер, причина падения.
Правило разрешения конфликтов, если клиент подходит под несколько групп.

### 7.1. Кого в кампанию не берём и почему

## 8. Что проверили и что не подтвердилось

Гипотезы, которые вы проверили и отвергли. По одной строке на гипотезу,
с числом.

## 9. Сохранение результатов

Три файла: очищенная таблица для второго ноутбука, журнал изменений,
итоговый список кампании.

In [17]:
# clean.to_parquet("clean_data.parquet", index=False)
# cleaning_log.to_csv("cleaning_log.csv", index=False, encoding="utf-8")
# campaign[["client_id", "group"]].to_csv("campaign_list.csv", index=False, encoding="utf-8")